# Day 20: FDE Professional Skills 2
**Handling ambiguity and customer-embedded problem solving, through simulations and role plays**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## Simulation: "The dashboard is slow"
A customer says the dashboard is slow. We have one week of request logs. Instead of guessing, we write a problem statement, list hypotheses, and let the data decide.

Timebox: **60 minutes**.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# pandas and matplotlib are already installed in Colab.

# ===== Option B: VS Code / local =====
# pip install pandas matplotlib
import pandas as pd, numpy as np, matplotlib.pyplot as plt
print("Ready. Start the 60 minute timer.")

## 2. The request logs

In [ ]:
rng = np.random.default_rng(20)
n = 20000
endpoints = ["/dashboard/summary", "/dashboard/charts", "/reports/export", "/users/me", "/orders"]
tenants = ["Bengaluru", "Tokyo", "Osaka", "Pune", "Chennai"]
# Traffic is busier during office hours, with a peak when everyone logs in at 9
hour_weights = np.array([1, 1, 1, 1, 1, 1, 2, 4, 8, 16, 14, 9, 7, 8, 8, 7, 6, 5, 3, 2, 2, 1, 1, 1], dtype=float)
hours = rng.choice(24, n, p=hour_weights / hour_weights.sum())
logs = pd.DataFrame({
    "ts": pd.Timestamp("2026-11-16") + pd.to_timedelta(rng.integers(0, 7, n), unit="D")
          + pd.to_timedelta(hours, unit="h") + pd.to_timedelta(rng.integers(0, 60, n), unit="min"),
    "endpoint": rng.choice(endpoints, n, p=[0.3, 0.3, 0.1, 0.15, 0.15]),
    "tenant": rng.choice(tenants, n),
    "page_size": rng.choice([20, 50, 100, 1000], n, p=[0.45, 0.3, 0.15, 0.1]),
    "region": rng.choice(["india-central", "japan-east"], n),
})
base = {"/dashboard/summary": 300, "/dashboard/charts": 450, "/reports/export": 900, "/users/me": 80, "/orders": 250}
ms = np.array([base[e] for e in logs["endpoint"]], dtype=float) * rng.uniform(0.7, 1.3, n)
# The hidden cause: large exports during the morning peak
hidden = (logs["endpoint"] == "/reports/export") & (logs["page_size"] == 1000) & (logs["ts"].dt.hour.between(9, 10))
ms[hidden.values] *= rng.uniform(8, 14, hidden.sum())
ms[(logs["tenant"] == "Osaka").values & (logs["endpoint"] == "/reports/export").values] *= 1.3
logs["duration_ms"] = ms.round(0)
logs.head()

## 3. Problem statement and hypotheses (write before analysing)

In [ ]:
problem = {
    "who": "Operations team using the dashboard",
    "what": "Pages feel slow; users report waiting many seconds",
    "impact": "Lost time every morning and declining trust in the tool",
    "success": "95th percentile response under 1 second for dashboard pages",
}
hypotheses = [
    "H1: Everything is slow in one region",
    "H2: One endpoint is much slower than the rest",
    "H3: Large page sizes cause the slowness",
    "H4: It is slow at a particular time of day",
    "H5: One tenant has much heavier usage",
]
for k, v in problem.items():
    print(f"{k:>8}: {v}")
print()
print("\n".join(hypotheses))

## 4. Test each hypothesis with evidence
We use the 95th percentile (p95): the time within which 95% of requests finish. Averages hide the painful slow requests.

In [ ]:
p95 = lambda s: s.quantile(0.95)

def test(col):
    return logs.groupby(col)["duration_ms"].agg(requests="count", p95=p95, median="median").round(0).sort_values("p95", ascending=False)

print("H1 by region:\n", test("region"), "\n")
print("H2 by endpoint:\n", test("endpoint"))

In [ ]:
print("H3 by page size:\n", test("page_size"), "\n")
logs["hour"] = logs["ts"].dt.hour
by_hour = test("hour").sort_index()
print("H4 slowest hours:\n", by_hour.sort_values("p95", ascending=False).head(3), "\n")
print("H5 by tenant (exports only):\n", logs[logs.endpoint == "/reports/export"].groupby("tenant")["duration_ms"].agg(p95=p95).round(0).sort_values("p95", ascending=False))

## 5. Narrow down: combine the strongest signals

In [ ]:
exports = logs[logs.endpoint == "/reports/export"]
combo = (exports.assign(morning=exports.hour.between(9, 10), big=exports.page_size == 1000)
         .groupby(["big", "morning"])["duration_ms"].agg(requests="count", p95=p95).round(0))
print(combo)

slow = logs[logs.duration_ms > 5000]
share = (slow.endpoint == "/reports/export").mean() * 100
print(f"\nRequests over 5 s: {len(slow)} ({len(slow) / len(logs) * 100:.1f}% of all). Exports make up {share:.0f}% of them.")

## 6. One chart for the readout

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
by_hour["p95"].plot(ax=ax, marker="o", color="#0FA3A3", label="All endpoints")
exp_hour = exports.groupby("hour")["duration_ms"].quantile(0.95)
exp_hour.plot(ax=ax, marker="o", color="#FF8A3D", label="/reports/export")
worst_hour = int(exp_hour.idxmax())
ax.set_title(f"Export requests spike around {worst_hour}:00, driving the slowness")
ax.set_xlabel("Hour of day"); ax.set_ylabel("p95 duration (ms)"); ax.legend()
plt.tight_layout(); plt.show()

## 7. Verdict on each hypothesis (computed from the data)

In [ ]:
region = test("region")["p95"]
endpoint = test("endpoint")["p95"]
size = test("page_size")["p95"]
def verdict(ratio, label):
    return f"confirmed: {label} ({ratio:.1f}x)" if ratio >= 1.5 else f"rejected: little difference ({ratio:.1f}x)"

exp_by_hour = exports.groupby("hour")["duration_ms"].quantile(0.95)
exp_by_tenant = exports.groupby("tenant")["duration_ms"].quantile(0.95)
verdicts = {
    "H1 region": verdict(region.max() / region.min(), f"{region.idxmax()} slowest"),
    "H2 endpoint": verdict(endpoint.max() / endpoint.drop(endpoint.idxmax()).max(), f"{endpoint.idxmax()} vs next slowest"),
    "H3 page size": verdict(size.max() / size.drop(size.idxmax()).max(), f"page_size {size.idxmax()} vs next"),
    "H4 time of day": verdict(exp_by_hour.max() / exp_by_hour.median(), f"exports at {int(exp_by_hour.idxmax())}:00 vs a typical hour"),
    "H5 tenant": verdict(exp_by_tenant.max() / exp_by_tenant.median(), f"{exp_by_tenant.idxmax()} exports vs a typical tenant"),
}
for h, v in verdicts.items():
    print(f"{h:<14} {v}")

### TODO: Write your readout
Answer first, then evidence with numbers, then a recommendation and the smallest next step with an owner and a date.

In [ ]:
readout = """
Finding:
Evidence 1:
Evidence 2:
Recommendation:
Next step (owner, date):
"""
print(readout)

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)